In [1]:
import json, os, itertools
from typing import List, Dict

In [37]:
dataset = "irb"
llm_model = "gemini-2.5-pro"
retrieval_model = "text-embedding-3-small"
do_retrieval = 1
use_chunk = 0
dataset_date = "30Oct2025"


work_dir = "/scratch/lamdo/IRB"
qa_metadata_path = "/scratch/lamdo/IRB/qa_metadata"

In [38]:
def metadata_folder_name_creation(dataset: str, llm_model_name: str, retrieval_model: str, use_retrieval_contexts: bool, use_chunk: bool):
    temp = [dataset, llm_model_name] + \
            ([retrieval_model, f"chunk{use_chunk}"] if use_retrieval_contexts else [])
    print(temp)
    return "__".join(temp)

def data_relative_path(dataset_name, dataset_date = None):
    if dataset_date:
        return os.path.join("benchmarks", dataset_date, dataset_name)
    else:
        raise NotImplemented
    

def filter_by_num_keypoints(att: Dict, choice: str = "single"):
    assert choice in ["single", "multi"]
    nkp = "single" if att.get("num_keypoints") == 1 else "multi"

    return nkp == choice


def filter_by_language(att: Dict, choice: str = "english_only"):
    assert choice in ["english_only", "multilingual"]

    langs = (att.get("evidence_attr").get("langs"))
    langs = list(itertools.chain.from_iterable(langs))

    _type = None
    if any([l != "en" for l in langs]):
        _type = "multilingual"
    else: _type = "english_only"

    return _type == choice


def filter_by_freshness(att: Dict, choice: int = 2024):
    create_timestamp = att.get("wiki_create_timestamp")
    published_dates = att.get("evidence_attr", {}).get("published_dates")

    create_timestamp = int(create_timestamp[:4])
    published_dates = [int(item[:4]) for item in list(itertools.chain.from_iterable(published_dates))]

    all_years = published_dates + [create_timestamp]

    min_year = min(all_years)

    return min_year == choice


def filter_by_topic(att: Dict, choice: str = "History_and_Society"):
    topics = att.get("topics")

    return any([choice in top for top in topics])


def filter_by_numhop(att: Dict, choice: int = "single"):
    assert choice in ["single", "multi"]
    nh = "single" if att.get("num_hops") == 1 else "multi"

    return nh == choice


def general_filter_func(att: Dict, choice_dict: Dict[str, str]):
    # the keys are 'language', 'freshness', 'topic', 'keypoints'

    filter_mapper = {
        "language": filter_by_language,
        "freshness": filter_by_freshness,
        "topic": filter_by_topic,
        "keypoints": filter_by_num_keypoints,
        "numhops": filter_by_numhop
    }

    return all([filter_mapper[k](att, v) for k, v in choice_dict.items()])

In [39]:
metadata_input_folder = os.path.join(qa_metadata_path, metadata_folder_name_creation(dataset, llm_model, retrieval_model, do_retrieval, use_chunk))

with open(os.path.join(metadata_input_folder, "eval_metadata.json")) as f:
    eval_metadata = json.load(f)

with open(os.path.join(metadata_input_folder, "eval_result.json")) as f:
    eval_result = json.load(f)

['irb', 'gemini-2.5-pro', 'text-embedding-3-small', 'chunk0']


In [40]:
benchmark_path = os.path.join(work_dir, data_relative_path(dataset, dataset_date))

attributes = []
with open(os.path.join(benchmark_path, "attributes.jsonl")) as f:
    for line in f:
        attributes.append(json.loads(line))

queries = {}
with open(os.path.join(benchmark_path, "queries.jsonl")) as f:
    for line in f:
        jline = json.loads(line)
        _id = jline["_id"]
        queries[_id] = jline

answers = {}
with open(os.path.join(benchmark_path, "answers.jsonl")) as f:
    for line in f:
        jline = json.loads(line)
        _id = jline["_id"]
        answers[_id] = jline

In [41]:
data_relative_path(dataset, dataset_date)

'benchmarks/30Oct2025/irb'

In [42]:
config = {"keypoints": "single"}

to_view = []
for att in attributes:
    query_id = att["_id"]
    if query_id not in eval_result: continue
    to_view.append([query_id, queries[query_id]["text"], eval_metadata["predictions"][query_id], answers[query_id]["short"], answers[query_id]["text"], eval_result[query_id], att])

In [43]:
with open(metadata_folder_name_creation(dataset, llm_model, retrieval_model, do_retrieval, use_chunk) + "_gitig_.json", "w") as f:
    json.dump(to_view, f, indent = 4)

['irb', 'gemini-2.5-pro', 'text-embedding-3-small', 'chunk0']
